In [0]:
# ============================================================
# 01_Bronze_Layer | CELL 1 — Config (FIXED PATHS)
# ============================================================
# KEY FIX: Spark needs "file:///" prefix to use the TRUE
# local filesystem. Without it, even /tmp/ gets routed
# through DBFS which is now disabled in your runtime.
# Python's os module uses /tmp/ directly (no prefix needed).
# Spark's read/write needs file:///tmp/ explicitly.
# Two different tools, two different path conventions.
# ============================================================

import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, lit, col

spark = SparkSession.builder.getOrCreate()

# ── Python paths (for os.makedirs, open(), etc.) ──────────
BASE_PATH_LOCAL   = "/tmp/fraud_detection"
LANDING_LOCAL     = f"{BASE_PATH_LOCAL}/landing"

# ── Spark paths (file:/// forces TRUE local filesystem) ───
BRONZE_SPARK      = "file:///tmp/fraud_detection/bronze/transactions"
CHECKPOINT_SPARK  = "file:///tmp/fraud_detection/checkpoints/bronze"

# ── Create folders using Python (always works) ────────────
for path in [
    f"{BASE_PATH_LOCAL}/landing",
    f"{BASE_PATH_LOCAL}/bronze/transactions",
    f"{BASE_PATH_LOCAL}/checkpoints/bronze",
]:
    os.makedirs(path, exist_ok=True)

TRANSACTION_SCHEMA = StructType([
    StructField("transaction_id",    StringType(),  False),
    StructField("customer_id",       StringType(),  False),
    StructField("amount",            DoubleType(),  True),
    StructField("currency",          StringType(),  True),
    StructField("merchant_category", StringType(),  True),
    StructField("country_code",      StringType(),  True),
    StructField("hour_of_day",       IntegerType(), True),
    StructField("login_attempts",    IntegerType(), True),
    StructField("transaction_dt",    StringType(),  True),
    StructField("is_fraud",          IntegerType(), True),
    StructField("ingestion_ts",      StringType(),  True),
])

print("✅ Config ready!")
print(f"   🐍 Python Base  : {BASE_PATH_LOCAL}")
print(f"   ⚡ Spark Bronze : {BRONZE_SPARK}")

In [0]:
# ============================================================
# 01_Bronze_Layer | CELL 2 — Data Generator
# ============================================================
# NO CHANGE needed here — pure Python, no Spark/DBFS involved
# ============================================================

import random
from datetime import datetime, timedelta

random.seed(42)

def generate_transaction(txn_id, day_offset=0):
    is_fraud = random.random() < 0.05
    if is_fraud:
        amount   = round(random.uniform(800, 9999), 2)
        hour     = random.choice([1, 2, 3, 4])
        country  = random.choice(["NG", "RO", "PK", "XX"])
        attempts = random.randint(3, 10)
    else:
        amount   = round(random.uniform(5, 500), 2)
        hour     = random.randint(8, 20)
        country  = random.choice(["US", "UK", "IN", "CA", "AU"])
        attempts = 1

    txn_date = datetime.now() - timedelta(days=day_offset)
    return {
        "transaction_id"   : f"TXN{txn_id:07d}",
        "customer_id"      : f"CUST{random.randint(1000, 9999)}",
        "amount"           : amount,
        "currency"         : "USD",
        "merchant_category": random.choice(["grocery","electronics","travel",
                                            "restaurant","online","atm"]),
        "country_code"     : country,
        "hour_of_day"      : hour,
        "login_attempts"   : attempts,
        "transaction_dt"   : txn_date.strftime("%Y-%m-%d %H:%M:%S"),
        "is_fraud"         : int(is_fraud),
        "ingestion_ts"     : datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

batch_1 = [generate_transaction(i, day_offset=1) for i in range(1, 1001)]
random.seed(99)
batch_2 = [generate_transaction(i, day_offset=0) for i in range(1001, 1501)]

print(f"✅ Batch 1 : {len(batch_1)} transactions | 🚨 Fraud: {sum(t['is_fraud'] for t in batch_1)}")
print(f"✅ Batch 2 : {len(batch_2)} transactions | 🚨 Fraud: {sum(t['is_fraud'] for t in batch_2)}")

In [0]:
# ============================================================
# 01_Bronze_Layer | CELL 3 — Python List → Spark DataFrame
# ============================================================
# KEY FIX: Instead of writing JSON to disk and reading back,
# we convert Python dicts DIRECTLY into Spark DataFrames.
# This completely bypasses the filesystem for data ingestion.
# WHY THIS IS ACTUALLY BETTER: In production, Kafka/Event Hub
# streams data directly into Spark the same way — no files!
# We're learning the more advanced pattern without realizing it.
# ============================================================

import pandas as pd

# Convert Python list → Pandas → Spark (with our schema)
batch1_df = spark.createDataFrame(pd.DataFrame(batch_1), schema=TRANSACTION_SCHEMA)
batch2_df = spark.createDataFrame(pd.DataFrame(batch_2), schema=TRANSACTION_SCHEMA)

# Add metadata columns
batch1_df = batch1_df.withColumn("batch_id", lit("batch_1")) \
                     .withColumn("load_ts",  current_timestamp())

batch2_df = batch2_df.withColumn("batch_id", lit("batch_2")) \
                     .withColumn("load_ts",  current_timestamp())

print("✅ DataFrames created directly from Python — no file I/O needed!")
print(f"\n📊 Batch 1 Schema:")
batch1_df.printSchema()
print(f"   Batch 1 Records : {batch1_df.count():,}")
print(f"   Batch 2 Records : {batch2_df.count():,}")

In [0]:
# ============================================================
# 01_Bronze_Layer | CELL 3b — Clean Slate (Run Once Only)
# ============================================================
# WHAT  : Drop the table so we start fresh.
# WHY   : saveAsTable in append mode accumulates on re-runs.
#         In production, pipelines have deduplication logic.
#         For now, we reset manually — clean slate, clean data.
# ============================================================

spark.sql("DROP TABLE IF EXISTS fraud_db.bronze_transactions")
print("✅ Table dropped — ready for clean run!")
print("   Now run Cell 4 → Cell 5 → Cell 6 → Cell 7 in order.")

In [0]:

# Create database to keep things organized
spark.sql("CREATE DATABASE IF NOT EXISTS fraud_db")
spark.sql("USE fraud_db")

batch1_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_db.bronze_transactions")    # ← No path, just table name!

count = spark.table("fraud_db.bronze_transactions").count()
print(f"✅ Bronze Layer — Batch 1 Written as Managed Table!")
print(f"   📦 Records in Bronze : {count:,}")
print(f"   📋 Table             : fraud_db.bronze_transactions")

In [0]:

batch2_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("fraud_db.bronze_transactions")    # ← Same table, append mode

total = spark.table("fraud_db.bronze_transactions").count()
print(f"✅ Batch 2 Appended!")
print(f"   📦 Total in Bronze : {total:,}")
print(f"   {'✅ Correct!' if total == 1500 else '❌ Check append logic'}")

In [0]:

bronze_df = spark.table("fraud_db.bronze_transactions")   # ← Read by table name

total = bronze_df.count()
fraud = bronze_df.filter(col("is_fraud") == 1).count()
legit = bronze_df.filter(col("is_fraud") == 0).count()

print("=" * 55)
print("  🪨 BRONZE LAYER — HEALTH REPORT")
print("=" * 55)
print(f"\n  📦 Total Records : {total:,}")
print(f"  🚨 Fraud         : {fraud:,}  ({round(fraud/total*100,1)}%)")
print(f"  ✅ Legitimate    : {legit:,}  ({round(legit/total*100,1)}%)")

print("\n  📋 Records by Batch:")
bronze_df.groupBy("batch_id").count().show()

print("  🌍 Top Countries:")
bronze_df.groupBy("country_code").count() \
         .orderBy("count", ascending=False).show(8)

print("  💰 Amount Stats:")
bronze_df.selectExpr(
    "round(min(amount),2) as min_amount",
    "round(max(amount),2) as max_amount",
    "round(avg(amount),2) as avg_amount"
).show()

In [0]:
from delta.tables import DeltaTable

# Use forName() instead of forPath() for managed tables
dt = DeltaTable.forName(spark, "fraud_db.bronze_transactions")

print("⏰ DELTA TABLE VERSION HISTORY:")
display(dt.history())

In [0]:
# Time Travel — read Version 0 (only Batch 1)
bronze_v0 = spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .table("fraud_db.bronze_transactions")       # ← .table() not .load()

current = spark.table("fraud_db.bronze_transactions").count()

print(f"⏪ Version 0 (Batch 1 only) : {bronze_v0.count():,} records")
print(f"✅ Current version          : {current:,} records")
print(f"\n🔥 Time Travel works — Delta Lake fully operational!")